In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Customer Churn Prediction

Compare Logistic Regression, Decision Tree, and Random Forest models. Keep the holdout test set at its original class distribution so evaluation reflects real data.

## 1. Load and inspect data

In [ ]:
df = pd.read_csv("../data/dataset-training.csv")
print("Shape:", df.shape)
display(df.head())
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 2. Clean and explore

In [ ]:
df = df.dropna(how="all").copy()
df = df.drop(columns=["CustomerID"], errors="ignore")
df = df.dropna(subset=["Churn"]).copy()

print("Churn counts:")
print(df["Churn"].value_counts())
print("\nChurn proportions:")
print(df["Churn"].value_counts(normalize=True).round(3))

In [ ]:
df["Churn"].value_counts().plot(kind="bar")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Customers")
plt.xticks(rotation=0)
plt.show()

In [ ]:
for feature in ["Subscription Type", "Contract Length"]:
    pd.crosstab(df[feature], df["Churn"], normalize="index").plot(kind="bar", stacked=True)
    plt.title(f"Churn Rate by {feature}")
    plt.xlabel(feature)
    plt.ylabel("Proportion")
    plt.xticks(rotation=0)
    plt.show()

df.boxplot(column="Payment Delay", by="Churn")
plt.title("Payment Delay by Churn")
plt.suptitle("")
plt.xlabel("Churn")
plt.ylabel("Payment Delay")
plt.show()

## 3. Split the data

Split before fitting preprocessing. Stratification preserves class proportions. Handle imbalance with model class weights rather than duplicating or discarding training rows.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("\nHoldout distribution:")
print(y_test.value_counts(normalize=True).round(3))

## 4. Preprocessing

Imputation, scaling, and one-hot encoding are fitted inside each model pipeline using training data only.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone

categorical_features = X_train.select_dtypes(include=["object", "category"]).columns
numeric_features = X_train.select_dtypes(include=["number"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 5. Train three models

Class weights help models learn from imbalanced classes without changing the holdout distribution.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=15, class_weight="balanced_subsample",
        random_state=42, n_jobs=-1
    )
}

pipelines = {}
for name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", estimator)
    ])
    pipeline.fit(X_train, y_train)
    pipelines[name] = pipeline
    print("Trained:", name)

## 6. Holdout evaluation

Compare accuracy, precision, recall, and F1 score. For churn, recall is important because it measures how many actual churners are detected.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

results = []
holdout_predictions = {}
for name, pipeline in pipelines.items():
    prediction = pipeline.predict(X_test)
    holdout_predictions[name] = prediction
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "Recall": recall_score(y_test, prediction, zero_division=0),
        "F1 score": f1_score(y_test, prediction, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1 score", ascending=False)
display(results_df.round(3))

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = pipelines[best_model_name]
best_prediction = holdout_predictions[best_model_name]

print("Best model by holdout F1 score:", best_model_name)
print(classification_report(y_test, best_prediction, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, best_prediction)
plt.title(best_model_name + " - Holdout Confusion Matrix")
plt.show()

## 7. Evaluate on the provided original test file

Treat this as a separate generalization check. Compare its target distribution with the training data. Do not tune model settings against this final evaluation set.

In [ ]:
test_df = pd.read_csv("../data/dataset-testing.csv")
test_df = test_df.dropna(how="all").dropna(subset=["Churn"]).copy()
X_external = test_df.drop(columns=["CustomerID", "Churn"], errors="ignore")
X_external = X_external.reindex(columns=X_train.columns)
y_external = test_df["Churn"]

print("Training churn distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("\nExternal test churn distribution:")
print(y_external.value_counts(normalize=True).round(3))
print("\nExternal feature shape:", X_external.shape)

In [ ]:
external_results = []
for name, pipeline in pipelines.items():
    prediction = pipeline.predict(X_external)
    external_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_external, prediction),
        "Precision": precision_score(y_external, prediction, zero_division=0),
        "Recall": recall_score(y_external, prediction, zero_division=0),
        "F1 score": f1_score(y_external, prediction, zero_division=0)
    })

external_results_df = pd.DataFrame(external_results).sort_values("F1 score", ascending=False)
display(external_results_df.round(3))

## 8. Interpretation

- Holdout metrics compare models on data sampled from the training dataset.
- External test metrics show how the models perform on the separate supplied dataset.
- A large performance gap can indicate distribution shift, data quality differences, or overfitting.
- Near-perfect scores are not proof of a strong model. Validate on representative future data and investigate possible leakage.

## 9. Save the selected model

The saved pipeline includes preprocessing and the classifier, so it accepts raw customer feature columns at prediction time.

In [ ]:
import joblib
from pathlib import Path

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "best_churn_model.pkl"
joblib.dump(best_model, model_path)
print("Saved model to:", model_path)